In this activity set, students will learn and utilize
the K-means clustering algorithm to identify the geographic patterns in cuisine types using data from Yelp. Their goal is to improve restaurant recommendation results and provide customers with
more flexible choices. As a first step, students will be instructed to select features for analysis from
a variety of domains, and in this case, the features of location and most popular culinary categories
are of greatest value. Next, students will transform the category data to 0-1 vectors for calculating
the closeness/similarity distance between restaurants. Here, one challenge is that the distances
evaluated by location and category might differ significantly, which introduces bias during the
clustering process. To avoid this unbalance, students will practice data scaling methods learned
in previous activities to rescale different features for clustering (e.g., min-max normalization).
Additionally, students will be educated with the concept of centroid and the detailed clustering
steps. Finally, students will examine the clustering results under different k values and explore the
dominant category of each cluster with plotting visualization.


In [ ]:
# Cell 1 — Test basic imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from IPython.display import display, clear_output

In [ ]:
# Cell 2 — Load the dataset and do a quick sanity check
url_full = "https://data.cityofnewyork.us/api/views/43nn-pn8j/rows.csv?accessType=DOWNLOAD"
df = pd.read_csv(url_full)

print(df.shape)
df.head()

(2922, 27)


,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,...,INSPECTION TYPE,Latitude,Longitude,Community Board,Council District,Census Tract,BIN,BBL,NTA,Location
0,50107669,CRAB DU JOUR,Brooklyn,605,AVENUE Z,11223,9172508158,Seafood,11/05/2025,Violations were cited in the following area(s).,...,Cycle Inspection / Re-inspection,40.586069,-73.971597,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-73.971597369805 40.586069315374)
1,50051965,LITTLE POLAND,Manhattan,200,2 AVENUE,10003,2127779728,Polish,01/08/2026,Violations were cited in the following area(s).,...,Cycle Inspection / Re-inspection,40.731268,-73.985712,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-73.985711695135 40.731267968726)
2,41721903,ARMONIE,Manhattan,1649,PARK AVENUE,10035,2127226400,Italian,07/02/2025,Violations were cited in the following area(s).,...,Cycle Inspection / Initial Inspection,40.799791,-73.942856,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-73.942856343025 40.799791221673)
3,40550548,GOODY'S BBQ,Queens,70-18,AMSTEL BOULEVARD,11692,7183189616,Caribbean,06/11/2024,Violations were cited in the following area(s).,...,Cycle Inspection / Initial Inspection,40.592631,-73.799786,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-73.799786361635 40.592630711621)
4,50142412,$1.50 HOT PIZZA,Brooklyn,255,LIVINGSTON STREET,11217,3478845060,Pizza,09/10/2025,Violations were cited in the following area(s).,...,Cycle Inspection / Re-inspection,40.688737,-73.983438,NaN,NaN,NaN,NaN,NaN,NaN,POINT (-73.983438342441 40.688737457989)


In [ ]:
# Cell 3 — Keep the latest inspection record per restaurant (CAMIS)

clean = df.copy()
# Sort so that the most recent inspection for each restaurant is last
clean = clean.sort_values(["CAMIS", "INSPECTION DATE"])

# For each CAMIS (restaurant id), keep only the last row (latest inspection)
rest_latest = (
    clean.dropna(subset=["CAMIS"])
        .groupby("CAMIS", as_index=False)
        .tail(1)
        .reset_index(drop=True)
)

# Sanity checks to confirm we have one row per restaurant
print("Latest rows:", rest_latest.shape)
print("Any duplicate CAMIS?", rest_latest["CAMIS"].duplicated().any())
print("Missing SCORE?", rest_latest["SCORE"].isna().mean())
print("Number of restaurants:", rest_latest["CAMIS"].nunique())

# Preview the fields we will use later for modeling and interpretation
rest_latest[["CAMIS","DBA","BORO","CUISINE DESCRIPTION","INSPECTION DATE","SCORE","GRADE","Latitude","Longitude"]].head()


Latest rows: (2790, 27)
Any duplicate CAMIS? False
Missing SCORE? 0.00035842293906810036
Number of restaurants: 2790


,CAMIS,DBA,BORO,CUISINE DESCRIPTION,INSPECTION DATE,SCORE,GRADE,Latitude,Longitude
0,40364389,OLD TOWN BAR,Manhattan,American,02/26/2025,10.0,A,40.737595,-73.989647
1,40364691,THE GEORGIAN SUITE KITCHEN,Manhattan,Continental,01/14/2020,13.0,A,40.775578,-73.964443
2,40364715,OLD HOMESTEAD,Manhattan,American,03/23/2023,12.0,A,40.741372,-74.005038
3,40364956,THE NEW STARLING ATHLETIC CLUB OF THE BRONX,Bronx,American,05/12/2025,12.0,A,40.830610,-73.849647
4,40365627,JAHN'S,Queens,American,12/09/2024,12.0,A,40.749653,-73.885074


In [ ]:
# Cell 4 — Build features, run K-means, and create an interactive panel

# Build the feature matrix X for clustering (step 1/2/3) and keep df_used aligned with X
def build_feature_matrix_step(df_rest, step=1, top_n_cuisines=15):
    df_used = df_rest.copy()

    df_used["SCORE"] = pd.to_numeric(df_used["SCORE"], errors="coerce")

    df_used = df_used.dropna(subset=["Latitude", "Longitude"])
    df_used = df_used[(df_used["Latitude"] != 0) & (df_used["Longitude"] != 0)]

    if step in (2, 3):
        df_used = df_used.dropna(subset=["SCORE"])

    if step == 1:
        X_num = df_used[["Latitude", "Longitude"]].copy()
    else:
        X_num = df_used[["Latitude", "Longitude", "SCORE"]].copy()

    cuisine_cols = []
    if step == 3:
        df_used["CUISINE DESCRIPTION"] = df_used["CUISINE DESCRIPTION"].fillna("Unknown")

        top = df_used["CUISINE DESCRIPTION"].value_counts().head(top_n_cuisines).index
        df_used["cuisine_top"] = np.where(df_used["CUISINE DESCRIPTION"].isin(top),
                                          df_used["CUISINE DESCRIPTION"], "Other")

        X_cui = pd.get_dummies(df_used["cuisine_top"], prefix="cuisine")
        cuisine_cols = list(X_cui.columns)

        X = pd.concat([X_num, X_cui], axis=1)
    else:
        X = X_num

    return X, df_used, cuisine_cols


# Summarize clusters with size/share and a few representative restaurants
def cluster_and_summarize(df_used, labels, n_examples=3):
    out = df_used.copy()
    out["cluster"] = labels
    total_n = len(out)

    grp = out.groupby("cluster", as_index=False).agg(
        n_restaurants=("cluster", "size"),
        mean_score=("SCORE", "mean"),
        mean_lat=("Latitude", "mean"),
        mean_lon=("Longitude", "mean")
    )
    grp["pct"] = grp["n_restaurants"] / total_n

    if "CUISINE DESCRIPTION" in out.columns:
        def top3(series):
            vc = series.value_counts(normalize=True).head(3)
            return ", ".join([f"{idx}({p:.0%})" for idx, p in vc.items()])
        top_cui = out.groupby("cluster")["CUISINE DESCRIPTION"].apply(top3).reset_index(name="top_cuisines")
        grp = grp.merge(top_cui, on="cluster", how="left")

    if "DBA" in out.columns:
        mean_score_map = out.groupby("cluster")["SCORE"].mean()

        def pick_examples(sub):
            c = sub.name
            mu = mean_score_map.get(c, np.nan)

            if pd.isna(mu) or "SCORE" not in sub.columns:
                sample = sub.sample(n=min(n_examples, len(sub)), random_state=42)
            else:
                tmp = sub.copy()
                tmp["abs_dev"] = (tmp["SCORE"] - mu).abs()
                sample = tmp.sort_values("abs_dev").head(n_examples)

            def fmt_row(r):
                dba = str(r.get("DBA", "NA"))
                boro = str(r.get("BORO", "NA"))
                sc = r.get("SCORE", np.nan)
                sc_str = "NA" if pd.isna(sc) else f"{sc:.0f}"
                return f"{dba} ({boro}, SCORE={sc_str})"

            return "; ".join(sample.apply(fmt_row, axis=1))

        examples = out.groupby("cluster").apply(pick_examples).reset_index(name="examples")
        grp = grp.merge(examples, on="cluster", how="left")

    grp = grp.sort_values("n_restaurants", ascending=False).reset_index(drop=True)
    grp["pct"] = grp["pct"].map(lambda x: f"{x:.1%}")
    return grp


# Plot restaurant locations colored by cluster
def plot_map(df_used, labels, title, only_cluster="All"):
    if only_cluster != "All":
        mask = (labels == int(only_cluster))
        df_plot = df_used.loc[mask].copy()
        labels_plot = labels[mask]
    else:
        df_plot = df_used
        labels_plot = labels

    plt.figure(figsize=(7, 6))
    plt.scatter(df_plot["Longitude"], df_plot["Latitude"], c=labels_plot, s=6, alpha=0.6)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title(title if only_cluster == "All" else f"{title} | cluster={only_cluster}")
    plt.show()


# Compute a silhouette curve on a subsample to keep the UI responsive
def silhouette_curve_sampled(X_for_model, k_min=2, k_max=12, random_state=42, max_n=5000):
    n = X_for_model.shape[0]
    if n > max_n:
        rng = np.random.default_rng(random_state)
        idx = rng.choice(n, size=max_n, replace=False)
        X_use = X_for_model[idx]
        sampled = True
    else:
        X_use = X_for_model
        sampled = False

    scores = []
    ks = list(range(k_min, k_max + 1))
    for k in ks:
        km = KMeans(n_clusters=k, random_state=random_state, n_init="auto")
        labels = km.fit_predict(X_use)
        scores.append(silhouette_score(X_use, labels))
    return ks, scores, sampled, X_use.shape[0]


step_dd = widgets.Dropdown(
    options=[("Step 1: Geo only (Lat/Lon)", 1),
             ("Step 2: Geo + Score", 2),
             ("Step 3: + Cuisine one-hot (optional)", 3)],
    value=1,
    description="Step:",
    style={"description_width": "initial"}
)

k_slider = widgets.IntSlider(
    value=5, min=2, max=20, step=1,
    description="k (#clusters):",
    style={"description_width": "initial"}
)

scaling_toggle = widgets.ToggleButton(
    value=True,
    description="Scaling (StandardScaler)",
    style={"description_width": "initial"}
)

topn_slider = widgets.IntSlider(
    value=15, min=5, max=30, step=1,
    description="Top N cuisines:",
    style={"description_width": "initial"}
)

cui_weight = widgets.FloatSlider(
    value=1.0, min=0.0, max=1.0, step=0.05,
    description="Cuisine weight (α):",
    style={"description_width": "initial"}
)

show_curve_toggle = widgets.ToggleButton(
    value=False,
    description="Show silhouette curve (2~12)",
    style={"description_width": "initial"}
)

cluster_dd = widgets.Dropdown(
    options=["All"],
    value="All",
    description="Show cluster:",
    style={"description_width": "initial"}
)

out = widgets.Output()
_state = {"df_used": None, "labels": None, "title": ""}


def update_visibility(*args):
    is_step3 = (step_dd.value == 3)
    topn_slider.layout.display = "block" if is_step3 else "none"
    cui_weight.layout.display = "block" if is_step3 else "none"

step_dd.observe(update_visibility, names="value")
update_visibility()


def update_map_only(*args):
    if _state["df_used"] is None or _state["labels"] is None:
        return
    with out:
        plot_map(_state["df_used"], _state["labels"], _state["title"], only_cluster=cluster_dd.value)

cluster_dd.observe(update_map_only, names="value")


def run_panel(step, k, scaling_on, top_n_cuisines, cuisine_alpha, show_curve):
    with out:
        clear_output(wait=True)

        X, df_used, cuisine_cols = build_feature_matrix_step(
            rest_latest, step=step, top_n_cuisines=top_n_cuisines
        )

        n = len(X)
        if n < 3:
            print("Not enough data to cluster.")
            return
        if k >= n:
            print(f"k={k} is too large for n={n}.")
            return

        if scaling_on:
            X_scaled = StandardScaler().fit_transform(X)
        else:
            X_scaled = X.values

        if step == 3 and cuisine_cols:
            col_names = list(X.columns)
            cui_idx = [col_names.index(c) for c in cuisine_cols]
            X_scaled[:, cui_idx] = X_scaled[:, cui_idx] * cuisine_alpha

        km = KMeans(n_clusters=k, random_state=42, n_init="auto")
        labels = km.fit_predict(X_scaled)

        sil = silhouette_score(X_scaled, labels)
        print(f"Step={step}, k={k}, scaling={scaling_on}, silhouette={sil:.4f}")
        if step == 3:
            print(f"TopN cuisines={top_n_cuisines}, cuisine weight α={cuisine_alpha:.2f}")

        summary = cluster_and_summarize(df_used, labels, n_examples=3)
        display(summary)

        if show_curve:
            ks, scs, sampled, used_n = silhouette_curve_sampled(
                X_scaled, 2, 12, random_state=42, max_n=5000
            )
            plt.figure(figsize=(6, 4))
            plt.plot(ks, scs, marker="o")
            plt.axvline(k, linestyle="--")
            plt.title("Silhouette curve (k=2~12)")
            plt.xlabel("k (# clusters)")
            plt.ylabel("silhouette score")
            plt.show()

            best_k = ks[int(np.argmax(scs))]
            note = f"(sampled n={used_n})" if sampled else "(full data)"
            print(f"(ref) best k in 2~12 = {best_k}, score={max(scs):.4f} {note}")

        cluster_dd.options = ["All"] + [str(i) for i in range(k)]
        cluster_dd.value = "All"

        title = f"NYC Restaurants Clusters (Step={step}, k={k}, scaling={scaling_on})"
        plot_map(df_used, labels, title, only_cluster="All")

        _state["df_used"] = df_used
        _state["labels"] = labels
        _state["title"] = title


ui = widgets.VBox([
    step_dd,
    k_slider,
    scaling_toggle,
    topn_slider,
    cui_weight,
    show_curve_toggle,
    cluster_dd,
])

def _on_change(change):
    run_panel(
        step=step_dd.value,
        k=k_slider.value,
        scaling_on=scaling_toggle.value,
        top_n_cuisines=topn_slider.value,
        cuisine_alpha=cui_weight.value,
        show_curve=show_curve_toggle.value
    )

for w in [step_dd, k_slider, scaling_toggle, topn_slider, cui_weight, show_curve_toggle]:
    w.observe(_on_change, names="value")

display(ui, out)
_on_change(None)


Output()